# Brecha digital y educación en Perú

Evaluación reproducible con los archivos ENAHO y ESCALE 2025 disponibles en el proyecto. Ejecuta las celdas en orden.

Semilla 42; separación por CONGLOME; escolares asistentes de 6 a 17 años con respuesta válida de uso de Internet. P314B y P316 se describen; P314D se usa en clustering con 0=no uso y se excluye de clasificación. P316B y P316C se exploran como predictores. ESCALE se vincula solo a UBIGEO.

In [ ]:
from pathlib import Path
import sys, pandas as pd
from IPython.display import display, Image
ROOT = Path.cwd()
if not (ROOT / 'src' / 'pipeline.py').exists(): ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))
from src.pipeline import build_dataset, run_clustering, run_supervised
from src.report import generate_report
print(ROOT)

## 1. Preparación y cruce
Cruza ENAHO 100, 200, 300 y Sumaria. Padron y matrícula escolar se enlazan por COD_MOD + ANEXO y se agregan por UBIGEO. La tabla resultante no identifica la escuela concreta de cada escolar.

In [ ]:
df = build_dataset()
print(f'{len(df):,} filas; {len(df.columns)} columnas; prevalencia={df.exclusion_digital.mean():.3f}')
display(df.head())
display(pd.read_csv(ROOT/'outputs/tables/diccionario_variables.csv').query('faltantes_pct > 0').sort_values('faltantes_pct', ascending=False).head(15))

## 2. Variables TIC ENAHO
Lugar (P314B) y propósito (P316) son descriptivos, pues se condicionan al uso de Internet. P314D entra al perfilamiento como ordinal (0=no uso; 1-4=frecuencia declarada), pero no a los predictores supervisados. P316B y P316C se incluyen como predictores exploratorios con tratamiento de faltantes dentro de cada fold.

In [ ]:
display(pd.read_csv(ROOT/'outputs/tables/resumen_variables_tic_enaho.csv'))

## 3. Perfiles no supervisados
Compara K-Means y Agglomerative para k=2 a 6, con Silhouette, Davies-Bouldin y Calinski-Harabasz. La muestra usa ingreso/gasto per cápita, razón alumnos/docente, frecuencia TIC P314D (0=no uso), retraso escolar y servicios básicos. Se excluye UBIGEO como identificador para que los perfiles no dependan solo del distrito. PCA muestra dos componentes; la varianza explicada se documenta en tabla. Los nombres son descriptivos y exploratorios, con medias para respaldarlos.

In [ ]:
run_clustering(df)
display(pd.read_csv(ROOT/'outputs/tables/metricas_clustering.csv').sort_values(['silhouette','davies_bouldin'],ascending=[False,True]))
display(pd.read_csv(ROOT/'outputs/tables/perfiles_clusters.csv'))
display(pd.read_csv(ROOT/'outputs/tables/pca_varianza.csv'))
display(Image(filename=str(ROOT/'outputs/figures/silhouette_clusters.png')))
display(Image(filename=str(ROOT/'outputs/figures/clusters_pca.png')))

## 4. Clasificación y prevención de fuga
GroupShuffleSplit aísla los conglomerados del test; la auditoría confirma cero grupos compartidos. GroupKFold reporta ROC-AUC, PR-AUC y F1 macro ponderados por fold. P307 y P314A definen población/resultado y no son predictores. P314B/P316 se reservan a descripción; P314D solo entra a clustering. Class_weight trata el desbalance y FACTOR07 pondera modelos y métricas cuando es admitido. La imputación, escalado y codificación se ajustan dentro de cada fold.

In [ ]:
run_supervised(df)
display(pd.read_csv(ROOT/'outputs/tables/comparacion_modelos.csv'))
display(pd.read_csv(ROOT/'outputs/tables/cv_groupkfold_modelos.csv'))
display(pd.read_json(ROOT/'outputs/tables/auditoria_split_conglomerado.json',typ='series'))
display(pd.read_csv(ROOT/'outputs/tables/distribucion_clases.csv'))
display(pd.read_csv(ROOT/'outputs/tables/shap_importancia_top10.csv'))
for name in ['evaluacion_modelos.png','shap_beeswarm.png','shap_dependence_1.png','shap_dependence_2.png','shap_waterfall_alto.png','shap_waterfall_bajo.png']:
    display(Image(filename=str(ROOT/'outputs/figures'/name)))

## 5. Informe y límites ESCALE
Padlocal.dbf se usa para contar locales por UBIGEO y cotejar CODLOCAL con Padron; no contiene coordenadas numéricas ni altitud. Los archivos disponibles no permiten calcular altitud, equipos informáticos ni conectividad escolar. Como ENAHO no contiene COD_MOD/CODLOCAL, no se asigna cada escolar a una escuela.

In [ ]:
display(pd.read_json(ROOT/'outputs/tables/auditoria_cruce_matricula_2025.json',typ='series'))
display(pd.read_json(ROOT/'outputs/tables/auditoria_cruce_padlocal.json',typ='series'))
print(generate_report())
print('El artículo PDF de 4–6 páginas se encuentra en outputs/reports; para regenerarlo ejecute python -m src.create_article_pdf.')